# Entrenamiento del modelo

Aquí comenzamos el entrenamiento de nuestro modelo, al que le apodamos "Pawl", en honor al pulpo del mundial del 2010 "Paul" 


In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib # Para guardar nuestro modelo final
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import log_loss, accuracy_score

import warnings
warnings.filterwarnings('ignore')

print("Librerías de Machine Learning importadas.")

Librerías de Machine Learning importadas.


Empezamos cargando los datos preprocesados

In [2]:
# Cargamos el dataset que exportamos anteriormente
ruta_datos = '../data/processed/historical_matches_elo.csv'
df = pd.read_csv(ruta_datos)

# Convertir fecha de nuevo a formato datetime
# Notamos que al exportar a CSV, las columnas de fechas siempre se vuelven de tipo texto
df['date'] = pd.to_datetime(df['date'])

print(f"Datos cargados: {df.shape[0]} partidos listos para entrenar.")


Datos cargados: 49257 partidos listos para entrenar.


### Creación de los datasets para Training y Test

Aquí dividimos el dataset en dos partes: Una para el entrenamiento
y el otro para testing.
Usaremos los datos de Qatar para Testing y los datos de entrenamiento será los datos previos a Qatar.

In [3]:

mask_test = (df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2022)
mask_train = df['date'] < '2022-11-20' # Fecha de inicio de Qatar 2022

# 2. Definir las características (X) y el objetivo (y)
features = ['home_elo', 'away_elo', 'elo_difference', 'home_advantage', 'is_friendly']

X_train = df.loc[mask_train, features]
y_train = df.loc[mask_train, 'target']
w_train = df.loc[mask_train, 'time_decay_weight'] # Nuestro peso temporal

X_test = df.loc[mask_test, features]
y_test = df.loc[mask_test, 'target']

print(f"Set de Entrenamiento: {X_train.shape[0]} partidos.")
print(f"Set de Prueba (Qatar 2022): {X_test.shape[0]} partidos.")

Set de Entrenamiento: 45698 partidos.
Set de Prueba (Qatar 2022): 64 partidos.


#### Training de Logistic Regression

In [4]:
print("Entrenando Regresión Logística (Modelo Base)...")

# Inicializar y entrenar (scikit-learn detecta automáticamente que es multiclase)
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train, sample_weight=w_train)

# Obtener probabilidades
logreg_probs = logreg.predict_proba(X_test)

print("¡Regresión Logística entrenada!")

Entrenando Regresión Logística (Modelo Base)...
¡Regresión Logística entrenada!


#### Training de XGBoost 
Hacemos Hyperparameter Tuning para el entrenamiento del modelo de XGBoost

In [5]:
print("Iniciando Búsqueda de Hiperparámetros (Validación Cruzada) para XGBoost...")
print("Esto puede tardar un par de minutos...\n")

# Definir la malla de búsqueda (Evitamos el sobreajuste con árboles pequeños)
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_lambda': [1, 5, 10],      # Regularización L2
    'n_estimators': [100, 200, 300]
}

# Configurar el buscador
xgb_base = xgb.XGBClassifier(
    objective='multi:softprob', 
    num_class=3, 
    eval_metric='mlogloss',
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,       # 10 combinaciones aleatorias
    scoring='neg_log_loss',
    cv=3,            # 3 Folds
    random_state=42,
    n_jobs=-1        # Usar todos los núcleos
)

# Entrenar
random_search.fit(X_train, y_train, sample_weight=w_train)

# Obtenemos el mejor modelo
xgb_best = random_search.best_estimator_
xgb_best_probs = xgb_best.predict_proba(X_test)

print(f"¡XGBoost optimizado con éxito!")
print(f"Mejores hiperparámetros encontrados: {random_search.best_params_}")

Iniciando Búsqueda de Hiperparámetros (Validación Cruzada) para XGBoost...
Esto puede tardar un par de minutos...

¡XGBoost optimizado con éxito!
Mejores hiperparámetros encontrados: {'reg_lambda': 5, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05}


### Evaluación

Procedemos a evaluar usando Log-Loss y Brier Score

In [6]:
# 1. Calcular Log-Loss
logloss_lr = log_loss(y_test, logreg_probs)
logloss_xgb = log_loss(y_test, xgb_best_probs)

# 2. Calcular Brier Score Multiclase (Manual porque sklearn solo tiene binario)
y_test_dummies = pd.get_dummies(y_test).values
brier_lr = np.mean(np.sum((logreg_probs - y_test_dummies)**2, axis=1))
brier_xgb = np.mean(np.sum((xgb_best_probs - y_test_dummies)**2, axis=1))

print("🏆 --- RESULTADOS DE EVALUACIÓN (QATAR 2022) --- 🏆")
print("-" * 50)
print(f"REGRESIÓN LOGÍSTICA:")
print(f"Log-Loss:    {logloss_lr:.4f}")
print(f"Brier Score: {brier_lr:.4f}")
print("-" * 50)
print(f"XGBOOST OPTIMIZADO:")
print(f"Log-Loss:    {logloss_xgb:.4f}")
print(f"Brier Score: {brier_xgb:.4f}")
print("-" * 50)

# Un pequeño recordatorio de por qué no usamos Accuracy
acc_xgb = accuracy_score(y_test, xgb_best.predict(X_test))
print(f"\n* Nota: El Accuracy del XGBoost es de {acc_xgb:.2%}, lo que demuestra")
print("* la alta varianza del fútbol y justifica el uso de Log-Loss.")

🏆 --- RESULTADOS DE EVALUACIÓN (QATAR 2022) --- 🏆
--------------------------------------------------
REGRESIÓN LOGÍSTICA:
Log-Loss:    1.0415
Brier Score: 0.6002
--------------------------------------------------
XGBOOST OPTIMIZADO:
Log-Loss:    1.0333
Brier Score: 0.5995
--------------------------------------------------

* Nota: El Accuracy del XGBoost es de 53.12%, lo que demuestra
* la alta varianza del fútbol y justifica el uso de Log-Loss.


In [8]:
import os

ruta_modelo = '../models/xgb_best_model.pkl'
os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)

# Exportar usando joblib
joblib.dump(xgb_best, ruta_modelo)

print(f"Modelo XGBoost guardado exitosamente en: {ruta_modelo}")
print("El Cuaderno 02 ha finalizado exitosamente. ¡Estamos listos para Monte Carlo!")

Modelo XGBoost guardado exitosamente en: ../models/xgb_best_model.pkl
El Cuaderno 02 ha finalizado exitosamente. ¡Estamos listos para Monte Carlo!
